RUNTIME: H100 preferred (A100/L4 also fine).

This notebook is aligned with the shared project framework:
- Uses repo-relative `data/` and `results/` paths
- Uses precomputed stratified fraction files from `00_eda.ipynb`
- Persists outputs incrementally after each `(fraction, seed)` run
- Supports resumable execution after Colab disconnects


In [ ]:
# 1) Install dependencies and resolve repo paths
!pip install -q -U "transformers<5" accelerate datasets scikit-learn pandas numpy

# If running this notebook directly in Colab (outside your cloned repo), run:
# !git clone https://github.com/phisomni-edu/cs4120-project /content/cs4120-project
# %cd /content/cs4120-project

import sys
from pathlib import Path
import torch

repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/cs4120-project'),
]
repo_root = None
for candidate in repo_candidates:
    if (candidate / 'src').exists() and (candidate / 'data').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError('Could not locate repo root with src/ and data/ directories.')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DATA_DIR = repo_root / 'data'
RESULTS_DIR = repo_root / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Repo root:', repo_root)
print('Data dir:', DATA_DIR)
print('Results dir:', RESULTS_DIR)


In [ ]:
# 2) Load data + labels + fraction file map
import ast
import gc
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import MultiLabelBinarizer

DATA_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
SEEDS = [42, 7, 21]

FRACTION_TAG = {
    0.01: '1pct',
    0.05: '5pct',
    0.10: '10pct',
    0.25: '25pct',
    0.50: '50pct',
    1.00: '100pct',
}

def parse_list_cell(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    try:
        out = ast.literal_eval(x)
        return out if isinstance(out, list) else []
    except Exception:
        return []

def choose_text_column(df):
    for col in ['text_clean_transformer', 'text']:
        if col in df.columns:
            return col
    raise ValueError('No supported text column found. Expected text_clean_transformer or text.')

TRAIN_FULL_PATH = DATA_DIR / 'train.csv'
VAL_PATH = DATA_DIR / 'val.csv'
TEST_PATH = DATA_DIR / 'test.csv'

if not TRAIN_FULL_PATH.exists() or not VAL_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError('Missing one of train.csv / val.csv / test.csv in data/. Run 00_eda.ipynb export first.')

train_full_df = pd.read_csv(TRAIN_FULL_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

text_col = choose_text_column(train_full_df)
label_col = 'label_names' if 'label_names' in train_full_df.columns else 'labels'

for df in [train_full_df, val_df, test_df]:
    df['labels_list'] = df[label_col].apply(parse_list_cell)
    if text_col in df.columns:
        df['text_model'] = df[text_col].fillna(df['text']).astype(str)
    else:
        df['text_model'] = df['text'].astype(str)

label_classes = sorted({lbl for labels in train_full_df['labels_list'] for lbl in labels})
if not label_classes:
    raise ValueError('No labels found after parsing. Check data/*.csv label formatting.')

print('Using label column:', label_col)
print('Text column:', text_col)
print('Num labels:', len(label_classes))
print('Sample labels:', label_classes[:8])

def fraction_train_path(data_fraction, seed):
    frac = float(data_fraction)
    seed = int(seed)
    if abs(frac - 1.0) < 1e-12:
        return DATA_DIR / 'train.csv'
    tag = FRACTION_TAG.get(frac)
    if tag is None or tag == '100pct':
        raise ValueError(f'Unsupported fraction: {data_fraction}')
    return DATA_DIR / f'train_{tag}_seed{seed}.csv'

for s in SEEDS:
    for f in DATA_FRACTIONS:
        p = fraction_train_path(f, s)
        if not p.exists():
            raise FileNotFoundError(f'Missing partition file: {p}')

print('All partition files found for configured fractions/seeds.')


In [ ]:
# 3) DistilBERT training/eval helpers
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)
from sklearn.metrics import f1_score
from src.evaluate import evaluate_run

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128
DEFAULT_PARAMS = {
    'learning_rate': 2e-5,
    'batch_size': 16,
    'num_epochs': 3,
    'weight_decay': 0.01,
}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
mlb = MultiLabelBinarizer(classes=label_classes)
mlb.fit([label_classes])

def prepare_model_df(df):
    out = df.copy()
    out['labels_binary'] = list(mlb.transform(out['labels_list']))
    return out

VAL_MODEL_DF = prepare_model_df(val_df)
TEST_MODEL_DF = prepare_model_df(test_df)

class EmotionDataset(Dataset):
    def __init__(self, df):
        self.encodings = tokenizer(
            df['text_model'].tolist(),
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
        self.labels = torch.tensor(np.stack(df['labels_binary']), dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx],
        }

def _compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        'f1_micro': f1_score(labels.astype(int), preds, average='micro', zero_division=0),
        'f1_macro': f1_score(labels.astype(int), preds, average='macro', zero_division=0),
    }

def load_train_fraction_df(data_fraction, seed):
    p = fraction_train_path(data_fraction, seed)
    df = pd.read_csv(p)
    df['labels_list'] = df[label_col].apply(parse_list_cell)
    if text_col in df.columns:
        df['text_model'] = df[text_col].fillna(df['text']).astype(str)
    else:
        df['text_model'] = df['text'].astype(str)
    return prepare_model_df(df)

def run_single_distilbert_experiment(data_fraction, seed, params=None):
    cfg = dict(DEFAULT_PARAMS)
    if params is not None:
        cfg.update(params)

    seed = int(seed)
    data_fraction = float(data_fraction)
    set_seed(seed)
    t0 = time.time()

    train_model_df = load_train_fraction_df(data_fraction, seed)
    train_ds = EmotionDataset(train_model_df)
    val_ds = EmotionDataset(VAL_MODEL_DF)
    test_ds = EmotionDataset(TEST_MODEL_DF)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_classes),
        problem_type='multi_label_classification',
    )

    args = TrainingArguments(
        output_dir=f'/tmp/distilbert_frac{int(data_fraction*100)}_seed{seed}',
        num_train_epochs=cfg['num_epochs'],
        per_device_train_batch_size=cfg['batch_size'],
        per_device_eval_batch_size=max(1, cfg['batch_size'] * 2),
        learning_rate=cfg['learning_rate'],
        weight_decay=cfg['weight_decay'],
        evaluation_strategy='epoch',
        save_strategy='no',
        logging_strategy='steps',
        logging_steps=50,
        report_to=[],
        seed=seed,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=_compute_metrics,
    )

    trainer.train()
    pred_out = trainer.predict(test_ds)
    logits_np = pred_out.predictions
    y_true = pred_out.label_ids.astype(int)
    y_pred = (torch.sigmoid(torch.tensor(logits_np)) >= 0.5).int().numpy()

    eval_out = evaluate_run(
        method='distilbert',
        data_fraction=data_fraction,
        seed=seed,
        label_names=[str(x) for x in label_classes],
        y_true=y_true,
        y_pred=y_pred,
    )

    elapsed_min = (time.time() - t0) / 60.0
    eval_out['overall']['train_rows'] = len(train_model_df)
    eval_out['overall']['learning_rate'] = cfg['learning_rate']
    eval_out['overall']['batch_size'] = cfg['batch_size']
    eval_out['overall']['num_epochs'] = cfg['num_epochs']
    eval_out['overall']['weight_decay'] = cfg['weight_decay']
    eval_out['overall']['elapsed_min'] = elapsed_min

    del trainer, model, train_ds, val_ds, test_ds, pred_out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return eval_out


In [ ]:
# 4) Crash-safe persistence + resumable execution
OVERALL_PATH = RESULTS_DIR / 'distilbert_overall.csv'
PER_CLASS_PATH = RESULTS_DIR / 'distilbert_per_class.csv'
README_PATH = RESULTS_DIR / 'distilbert_results.csv'
ERRORS_PATH = RESULTS_DIR / 'distilbert_errors.csv'

def _safe_read_csv(path):
    if not path.exists():
        return pd.DataFrame()
    try:
        if path.stat().st_size == 0:
            return pd.DataFrame()
        return pd.read_csv(path)
    except (pd.errors.EmptyDataError, FileNotFoundError):
        return pd.DataFrame()

overall_df = _safe_read_csv(OVERALL_PATH)
per_class_df = _safe_read_csv(PER_CLASS_PATH)
errors_df = _safe_read_csv(ERRORS_PATH)

def _dedup_overall(df):
    if df.empty:
        return df
    return (
        df.sort_values(['method', 'seed', 'data_fraction'])
          .drop_duplicates(subset=['method', 'seed', 'data_fraction'], keep='last')
          .reset_index(drop=True)
    )

def _dedup_per_class(df):
    if df.empty:
        return df
    return (
        df.sort_values(['method', 'seed', 'data_fraction', 'emotion'])
          .drop_duplicates(subset=['method', 'seed', 'data_fraction', 'emotion'], keep='last')
          .reset_index(drop=True)
    )

def _persist_results():
    global overall_df, per_class_df, errors_df

    overall_df = _dedup_overall(overall_df)
    per_class_df = _dedup_per_class(per_class_df)

    overall_df.to_csv(OVERALL_PATH, index=False)
    per_class_df.to_csv(PER_CLASS_PATH, index=False)

    if not per_class_df.empty:
        readme_df = per_class_df[['method', 'data_fraction', 'seed', 'emotion', 'f1', 'precision', 'recall']].copy()
        readme_df.to_csv(README_PATH, index=False)

    if not errors_df.empty:
        errors_df.to_csv(ERRORS_PATH, index=False)

def _completed_pairs():
    if overall_df.empty:
        return set()
    return set((float(r.data_fraction), int(r.seed)) for r in overall_df[['data_fraction', 'seed']].itertuples(index=False))

def show_distilbert_progress():
    done = _completed_pairs()
    expected = [(float(f), int(s)) for s in SEEDS for f in DATA_FRACTIONS]
    pending = [p for p in expected if p not in done]

    print(f'Completed runs: {len(done)} / {len(expected)}')
    if pending:
        print('Pending:', pending)
    else:
        print('No pending runs.')

    if not overall_df.empty:
        display(overall_df.sort_values(['seed', 'data_fraction']).tail(10))

def run_and_persist_one(data_fraction, seed, *, force=False, params=None):
    global overall_df, per_class_df, errors_df

    key = (float(data_fraction), int(seed))
    if (not force) and key in _completed_pairs():
        print(f'Skipping completed run: {key}')
        return

    print(f'Running DistilBERT: fraction={data_fraction}, seed={seed}')
    try:
        eval_out = run_single_distilbert_experiment(data_fraction, seed, params=params)

        if not overall_df.empty:
            overall_df = overall_df[~((overall_df['data_fraction'].astype(float) == float(data_fraction)) & (overall_df['seed'].astype(int) == int(seed)))]
        if not per_class_df.empty:
            per_class_df = per_class_df[~((per_class_df['data_fraction'].astype(float) == float(data_fraction)) & (per_class_df['seed'].astype(int) == int(seed)))]

        overall_df = pd.concat([overall_df, eval_out['overall']], ignore_index=True) if not overall_df.empty else eval_out['overall'].copy()
        per_class_df = pd.concat([per_class_df, eval_out['per_class']], ignore_index=True) if not per_class_df.empty else eval_out['per_class'].copy()

        _persist_results()
        macro_f1 = float(eval_out['overall'].iloc[0]['macro_f1'])
        print(f'Completed and saved: macro_f1={macro_f1:.4f}')

    except Exception as exc:
        err_row = pd.DataFrame([{'seed': int(seed), 'data_fraction': float(data_fraction), 'error': str(exc)}])
        errors_df = pd.concat([errors_df, err_row], ignore_index=True) if not errors_df.empty else err_row
        _persist_results()
        print(f'FAILED: {exc}')

def run_pending(*, fractions=None, seeds=None, max_runs=None, force=False, params=None):
    fracs = [float(f) for f in (fractions if fractions is not None else DATA_FRACTIONS)]
    seed_list = [int(s) for s in (seeds if seeds is not None else SEEDS)]

    planned = [(f, s) for s in seed_list for f in fracs]
    pending = [p for p in planned if force or p not in _completed_pairs()]

    if max_runs is not None:
        pending = pending[: int(max_runs)]

    print(f'Pending selected runs: {len(pending)}')
    for f, s in pending:
        run_and_persist_one(f, s, force=force, params=params)

    show_distilbert_progress()

show_distilbert_progress()


In [ ]:
# 5) Run patterns (use one at a time)
# Example A: run one job
# run_and_persist_one(0.01, 42)

# Example B: run one fraction across all seeds
# run_pending(fractions=[0.10])

# Example C: run only next 1-2 jobs (best for crash safety)
# run_pending(max_runs=2)

# Example D: continue all remaining jobs
run_pending()

show_distilbert_progress()
